# Preparação e Tratamento dos Dados

Nesta etapa, os datasets serão preparados para integração, corrigindo inconsistências identificadas durante a exploração inicial.

O processo inclui:

- conversão e padronização de tipos;
- tratamento de valores ausentes;
- agregação da temperatura para a granularidade `country + year`;
- remoção de colunas técnicas ou não analíticas;
- padronização dos nomes das áreas;
- validação das chaves de integração;
- integração das variáveis de clima e uso de pesticidas ao dataset principal;
- validação da estrutura final.

O objetivo é gerar um dataset integrado e consistente para a etapa posterior de análise e visualização.

In [ ]:
import pandas as pd

## 1. Carregamento dos dados

Os quatro datasets utilizados no projeto serão carregados novamente para os DataFrames que serão utilizados durante o tratamento.

In [ ]:
pesticides = pd.read_csv('../dados/pesticides.csv')
rainfall = pd.read_csv('../dados/rainfall.csv')
temp = pd.read_csv('../dados/temp.csv')
yield_data = pd.read_csv('../dados/yield.csv')

## 2. Diagnóstico inicial

Antes de realizar as transformações, serão verificadas as principais inconsistências identificadas durante a exploração inicial.

O objetivo é confirmar quais problemas precisam ser tratados e estabelecer uma referência para validar os resultados posteriormente.

In [ ]:
print("Pesticides:")
print(pesticides.isna().sum())

print("\nRainfall:")
print(rainfall.isna().sum())

print("\nTemp:")
print(temp.isna().sum())

print("\nYield:")
print(yield_data.isna().sum())

### 2.1 Tratamento de `rainfall`

A coluna `average_rain_fall_mm_per_year` apresenta valores que precisam ser convertidos para um tipo numérico.

Antes da conversão, será verificado se existem valores que não podem ser interpretados como números.

In [ ]:
rainfall[
    pd.to_numeric(
        rainfall['average_rain_fall_mm_per_year'],
        errors='coerce'
    ).isna()
]['average_rain_fall_mm_per_year'].value_counts(dropna=False)

In [ ]:
rainfall['average_rain_fall_mm_per_year'] = pd.to_numeric(
    rainfall['average_rain_fall_mm_per_year'],
    errors='coerce'
)

In [ ]:
print(
    'Valores ausentes após conversão:',
    rainfall['average_rain_fall_mm_per_year'].isna().sum()
)

## 3. Tratamento — Temp

O dataset `temp` possui múltiplas observações para algumas combinações de `country + year`.

Como o dataset principal possui granularidade anual por área, será necessário obter uma única temperatura média para cada combinação de país e ano.

Antes da agregação, será verificada a presença de valores ausentes.

In [179]:
print(
    'Valores ausentes antes da agregação:',
    temp['avg_temp'].isna().sum()
)

Valores ausentes antes da agregação: 2547


### 3.1 Agregação anual da temperatura

As múltiplas observações existentes para uma mesma combinação de país e ano serão agregadas pela média da temperatura.

Dessa forma, o dataset resultante terá granularidade `country + year`, compatível com os demais datasets utilizados na integração.

In [181]:
temp_anual = (
    temp
    .groupby(['country', 'year'], as_index=False)
    ['avg_temp']
    .mean()
)

print(
    temp_anual
    .groupby(['country', 'year'])
    .size()
    .value_counts()
    .sort_index()
)

1    28514
Name: count, dtype: int64


In [182]:
temp_anual['avg_temp'].isna().sum()

np.int64(1035)

## 4. Tratamento — yield_data

O dataset `yield_data` contém colunas técnicas e códigos utilizados na origem dos dados, além das variáveis necessárias para a análise.

Será mantido apenas o conjunto de colunas analíticas relevantes para o projeto.

In [ ]:
yield_clean = yield_data.drop(
    columns=[
        'Domain Code',
        'Domain',
        'Element Code',
        'Element',
        'Year Code',
        'Area Code',
        'Item Code'
    ]
)

In [ ]:
print(
    yield_clean
    .groupby(['Area', 'Item', 'Year'])
    .size()
    .value_counts()
    .sort_index()
)

## 5. Tratamento — Pesticides

O dataset de pesticidas será reduzido às colunas necessárias para a integração com o dataset principal.

A coluna `Value` será renomeada para `pesticides_tonnes` para representar explicitamente sua finalidade analítica.

In [ ]:
pesticides_clean = pesticides.rename(
    columns={'Value': 'pesticides_tonnes'}
)[
    ['Area', 'Year', 'pesticides_tonnes']
]

In [183]:
print(
    pesticides_clean
    .groupby(['Area', 'Year'])
    .size()
    .value_counts()
    .sort_index()
)

1    4349
Name: count, dtype: int64


## 6. Tratamento — Rainfall

O dataset de precipitação será padronizado para utilizar apenas as colunas necessárias à integração.

Também será realizada a limpeza dos espaços presentes nos nomes das colunas e das áreas.

In [ ]:
rainfall.columns = rainfall.columns.str.strip()

In [ ]:
rainfall_clean = rainfall.rename(
    columns={
        'average_rain_fall_mm_per_year': 'rainfall_mm'
    }
)[
    ['Area', 'Year', 'rainfall_mm']
]

rainfall_clean['Area'] = rainfall_clean['Area'].str.strip()

In [ ]:
print(
    rainfall_clean
    .groupby(['Area', 'Year'])
    .size()
    .value_counts()
    .sort_index()
)

In [ ]:
print(
    'Valores ausentes em rainfall:',
    rainfall_clean['rainfall_mm'].isna().sum()
)

## 7. Padronização das áreas

Os datasets utilizam diferentes nomenclaturas para algumas áreas geográficas.

Como a integração será realizada por `Area + Year`, diferenças de nomenclatura podem impedir a correspondência entre os datasets mesmo quando representam a mesma área.

As correções de nomenclatura foram definidas durante a exploração dos dados. Como cada fonte possui uma nomenclatura própria, serão utilizadas chaves específicas para `pesticides`, `rainfall` e `temp`.

In [ ]:
mapeamento_pesticides = {
    'Ethiopia PDR': 'Ethiopia',
    'Czechoslovakia': 'Czechia',
    'Sudan (former)': 'Sudan'
}

In [ ]:
mapeamento_rainfall = {
    'Russian Federation': 'Russia',
    'Iran (Islamic Republic of)': 'Iran',
    'Bolivia (Plurinational State of)': 'Bolivia',
    'Venezuela (Bolivarian Republic of)': 'Venezuela, RB',
    'Republic of Moldova': 'Moldova',
    'United Republic of Tanzania': 'Tanzania',
    'Viet Nam': 'Vietnam',
    'United States of America': 'United States',
    'Syrian Arab Republic': 'Syria',
    'Republic of Korea': 'South Korea',
    'Czechia': 'Czech Republic',
    'The former Yugoslav Republic of Macedonia': 'Macedonia'
}

In [ ]:
mapeamento_temp = {}

In [ ]:
yield_clean['Area_pesticides'] = yield_clean['Area'].replace(
    mapeamento_pesticides
)

In [ ]:
yield_clean['Area_rainfall'] = yield_clean['Area'].replace(
    mapeamento_rainfall
)

In [ ]:
yield_clean['Area_temp'] = yield_clean['Area'].replace(
    mapeamento_temp
)

## 8. Validação das áreas e chaves de integração

Após a padronização das nomenclaturas, será verificada a cobertura das chaves `Area + Year` do dataset principal (`yield`) nos datasets auxiliares.

A validação será realizada separadamente para `pesticides`, `rainfall` e `temp`, considerando as respectivas nomenclaturas padronizadas.

In [ ]:
chaves_yield = set(
    zip(yield_clean['Area'], yield_clean['Year'])
)

chaves_pesticides = set(
    zip(
        pesticides_clean['Area'],
        pesticides_clean['Year']
    )
)

chaves_rainfall = set(
    zip(
        rainfall_clean['Area'],
        rainfall_clean['Year']
    )
)

chaves_temp = set(
    zip(
        temp_anual['country'],
        temp_anual['year']
    )
)

chaves_yield_pesticides = set(
    zip(
        yield_clean['Area_pesticides'],
        yield_clean['Year']
    )
)

chaves_yield_rainfall = set(
    zip(
        yield_clean['Area_rainfall'],
        yield_clean['Year']
    )
)

chaves_yield_temp = set(
    zip(
        yield_clean['Area_temp'],
        yield_clean['Year']
    )
)

print('Chaves únicas no yield:', len(chaves_yield))
print('Chaves únicas no pesticides:', len(chaves_pesticides))
print('Chaves únicas no rainfall:', len(chaves_rainfall))
print('Chaves únicas no temp:', len(chaves_temp))

print(
    '\nYield x Pesticides:',
    len(chaves_yield_pesticides & chaves_pesticides),
    '/',
    len(chaves_yield_pesticides)
)

print(
    'Yield x Rainfall:',
    len(chaves_yield_rainfall & chaves_rainfall),
    '/',
    len(chaves_yield_rainfall)
)

print(
    'Yield x Temp:',
    len(chaves_yield_temp & chaves_temp),
    '/',
    len(chaves_yield_temp)
)

## 9. Integração dos dados

Os datasets auxiliares serão integrados ao dataset principal (`yield`) utilizando `Area + Year` como chave de integração.

Cada fonte utiliza sua respectiva nomenclatura padronizada de área:

- `Area_pesticides` para `pesticides`;
- `Area_rainfall` para `rainfall`;
- `Area_temp` para `temp`.

A integração será realizada preservando a granularidade original do dataset principal, sem multiplicação de registros.

In [ ]:
rainfall_chaves = (
    rainfall_clean
    .set_index(['Area', 'Year'])['rainfall_mm']
)

yield_integrado = yield_clean.copy()

chaves_yield = pd.MultiIndex.from_arrays(
    [
        yield_integrado['Area_rainfall'],
        yield_integrado['Year']
    ]
)

yield_integrado['rainfall_mm'] = chaves_yield.map(
    rainfall_chaves
)

print('Registros:', len(yield_integrado))
print('Valores de rainfall:', yield_integrado['rainfall_mm'].notna().sum())
print('Valores ausentes:', yield_integrado['rainfall_mm'].isna().sum())

In [ ]:
pesticides_chaves = (
    pesticides_clean
    .set_index(['Area', 'Year'])['pesticides_tonnes']
)

chaves_yield = pd.MultiIndex.from_arrays(
    [
        yield_integrado['Area_pesticides'],
        yield_integrado['Year']
    ]
)

yield_integrado['pesticides_tonnes'] = chaves_yield.map(
    pesticides_chaves
)

print('Registros:', len(yield_integrado))
print(
    'Valores de pesticides:',
    yield_integrado['pesticides_tonnes'].notna().sum()
)
print(
    'Valores ausentes:',
    yield_integrado['pesticides_tonnes'].isna().sum()
)

In [ ]:
temp_chaves = (
    temp_anual
    .set_index(['country', 'year'])['avg_temp']
)

chaves_yield = pd.MultiIndex.from_arrays(
    [
        yield_integrado['Area_temp'],
        yield_integrado['Year']
    ]
)

yield_integrado['avg_temp'] = chaves_yield.map(
    temp_chaves
)

print('Registros:', len(yield_integrado))
print(
    'Valores de temperatura:',
    yield_integrado['avg_temp'].notna().sum()
)
print(
    'Valores ausentes:',
    yield_integrado['avg_temp'].isna().sum()
)

## 10. Validação do dataset integrado

Após a integração dos datasets auxiliares, serão realizadas validações para verificar a consistência, granularidade e presença de valores ausentes.

A validação considera a preservação da granularidade `Area + Item + Year`, os tipos das variáveis e a presença de valores ausentes nas variáveis integradas.

In [ ]:
print('Registros:', len(yield_integrado))

print('\nDuplicidade Area + Item + Year:')
print(
    yield_integrado
    .groupby(['Area', 'Item', 'Year'])
    .size()
    .value_counts()
    .sort_index()
)

In [ ]:
print('Tipos das colunas:')
print(yield_integrado.dtypes)

print('\nValores ausentes:')
print(yield_integrado.isna().sum())

### Considerações da validação

O dataset integrado preservou a granularidade original de `Area + Item + Year`, sem duplicação de registros. As variáveis de produtividade e identificação não apresentam valores ausentes.

As variáveis auxiliares apresentam valores ausentes devido às diferenças de cobertura entre as fontes originais. Esses valores serão mantidos e tratados conforme a necessidade das análises realizadas nas etapas posteriores.

## 11. Exportação do dataset tratado

O dataset final será exportado para um novo arquivo, preservando os arquivos originais.

O arquivo `yield_integrado.csv` será utilizado nas etapas posteriores de análise e visualização.

In [ ]:
yield_integrado.to_csv(
    '../dados/yield_integrado.csv',
    index=False
)

print('Arquivo salvo com sucesso.')

## 12. Considerações finais

A etapa de preparação e tratamento resultou em um dataset integrado a partir das informações de produtividade agrícola, precipitação, uso de pesticidas e temperatura.

Durante o processo foram realizadas:

- conversão de tipos;
- tratamento da estrutura de `rainfall`;
- agregação das temperaturas para a granularidade anual;
- remoção de colunas técnicas de `yield_data`;
- seleção e renomeação de variáveis analíticas;
- padronização das nomenclaturas geográficas;
- validação das chaves `Area + Year`;
- integração dos datasets auxiliares;
- validação da granularidade e dos valores ausentes.

O dataset resultante será utilizado como base para a etapa de análise exploratória e geração dos indicadores do projeto.